# Fine-tune a pretrained sentence model for plagiarism checking

Starts from `all-MiniLM-L6-v2` (pretrained) and fine-tunes it on paraphrase pairs (PAWS + MRPC + QQP) so it recognises
**copied meaning, not just copied words**. Output: `semantic.zip` that you unzip into `models/semantic/` of the app.

**Before running (Kaggle settings, right sidebar):**
1. Accelerator -> **GPU T4 x2** (or P100). 2. Internet -> **On** (needs a phone-verified account) - used to download the model and datasets.
3. Run All. Expect roughly 15-25 minutes for 2 epochs.

Set `EPOCHS = 0` below to skip training and just export the plain pretrained model.

In [ ]:
!pip -q install sentence-transformers datasets

In [ ]:
import os, json, random, shutil, time
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
from transformers import get_linear_schedule_with_warmup

# ---------------- settings ----------------
BASE        = "sentence-transformers/all-MiniLM-L6-v2"
EPOCHS      = 2          # 0 = no fine-tuning, export the pretrained model as is
BATCH       = 64
LR          = 2e-5
MAX_LEN     = 64
QQP_TRAIN   = 40000      # QQP is huge; use a subset
SEED        = 42
OUT         = "/kaggle/working/semantic"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu" and EPOCHS > 0:
    print("WARNING: no GPU found - training will be very slow. Enable the GPU accelerator.")

## 1. Data (label 1 = same meaning / paraphrase, 0 = different)

In [ ]:
def to_df(ds, a, b):
    return pd.DataFrame({"s1": ds[a], "s2": ds[b], "label": ds["label"]})

parts = {"train": [], "val": [], "test": []}

# PAWS: sentence pairs with very high word overlap but sometimes different meaning (teaches the model not to trust overlap)
try:
    d = load_dataset("google-research-datasets/paws", "labeled_final")
    parts["train"].append(to_df(d["train"], "sentence1", "sentence2"))
    parts["val"].append(to_df(d["validation"], "sentence1", "sentence2"))
    parts["test"].append(to_df(d["test"], "sentence1", "sentence2"))
    print("PAWS ok")
except Exception as e:
    print("PAWS skipped:", e)

# MRPC: news paraphrases
try:
    d = load_dataset("nyu-mll/glue", "mrpc")
    parts["train"].append(to_df(d["train"], "sentence1", "sentence2"))
    parts["val"].append(to_df(d["validation"], "sentence1", "sentence2"))
    parts["test"].append(to_df(d["test"], "sentence1", "sentence2"))
    print("MRPC ok")
except Exception as e:
    print("MRPC skipped:", e)

# QQP: duplicate questions (subset, split into disjoint train/val/test)
try:
    d = load_dataset("nyu-mll/glue", "qqp")
    q = to_df(d["train"], "question1", "question2").sample(QQP_TRAIN + 4000, random_state=SEED).reset_index(drop=True)
    parts["train"].append(q.iloc[:QQP_TRAIN]); parts["val"].append(q.iloc[QQP_TRAIN:QQP_TRAIN+2000]); parts["test"].append(q.iloc[QQP_TRAIN+2000:])
    print("QQP ok")
except Exception as e:
    print("QQP skipped:", e)

assert parts["train"], "No dataset could be loaded - check that Internet is switched on."
train = pd.concat(parts["train"]).dropna().sample(frac=1, random_state=SEED).reset_index(drop=True)
val   = pd.concat(parts["val"]).dropna().reset_index(drop=True)
test  = pd.concat(parts["test"]).dropna().reset_index(drop=True)
test  = test[test.label.isin([0, 1])]
print(len(train), len(val), len(test), "| positive rate (train):", round(train.label.mean(), 3))

## 2. Evaluation helpers and baselines

In [ ]:
def embed(model, texts, bs=256):
    return model.encode(texts, batch_size=bs, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)

def model_scores(model, df):
    return (embed(model, df.s1.tolist()) * embed(model, df.s2.tolist())).sum(1)

def best_threshold(scores, y):
    grid = np.linspace(0.05, 0.95, 91)
    f1s = [f1_score(y, scores >= t) for t in grid]
    return float(grid[int(np.argmax(f1s))])

def evaluate(name, s_val, s_test):
    thr = best_threshold(s_val, val.label.values)
    y, p = test.label.values, s_test >= thr
    r = dict(model=name, threshold=round(thr, 2), accuracy=accuracy_score(y, p), precision=precision_score(y, p),
             recall=recall_score(y, p), f1=f1_score(y, p), auc=roc_auc_score(y, s_test))
    return {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}

results = []

# baseline 1: TF-IDF cosine (what the app uses today)
tf = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2)).fit(pd.concat([train.s1, train.s2]).tolist())
def tfidf_scores(df):
    A, B = tf.transform(df.s1), tf.transform(df.s2)
    return np.asarray(A.multiply(B).sum(axis=1)).ravel()
results.append(evaluate("TF-IDF cosine (lexical baseline)", tfidf_scores(val), tfidf_scores(test)))

# baseline 2: pretrained model, no fine-tuning
model = SentenceTransformer(BASE, device=device)
model.max_seq_length = MAX_LEN
results.append(evaluate("Pretrained MiniLM (no fine-tuning)", model_scores(model, val), model_scores(model, test)))
pd.DataFrame(results)

## 3. Fine-tune

In [ ]:
os.makedirs(OUT, exist_ok=True)
model.save(OUT)                       # always leave a usable model in OUT (the pretrained one if EPOCHS = 0)

def to_features(texts):
    # newer sentence-transformers versions return some non-tensor entries too: move only tensors to the GPU
    feats = model.tokenize(texts)
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in feats.items()}

def train_model():
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps_per_epoch = (len(train) + BATCH - 1) // BATCH
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps_per_epoch * EPOCHS), steps_per_epoch * EPOCHS)
    use_amp = device == "cuda"
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except (AttributeError, TypeError):   # older torch
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    best_auc = -1
    for epoch in range(EPOCHS):
        model.train()
        order = np.random.permutation(len(train)); t0 = time.time(); run = 0.0
        for step in range(steps_per_epoch):
            idx = order[step * BATCH:(step + 1) * BATCH]
            b = train.iloc[idx]
            fa = to_features(b.s1.tolist())
            fb = to_features(b.s2.tolist())
            y = torch.tensor(b.label.values, dtype=torch.float32, device=device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                ea = model(fa)["sentence_embedding"]; eb = model(fb)["sentence_embedding"]
            loss = F.mse_loss(F.cosine_similarity(ea.float(), eb.float()), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
            if (step + 1) % 200 == 0:
                print(f"epoch {epoch+1} step {step+1}/{steps_per_epoch} loss {run/200:.4f}"); run = 0.0
        s_val = model_scores(model, val)
        auc = roc_auc_score(val.label.values, s_val)
        print(f"== epoch {epoch+1} done in {time.time()-t0:.0f}s | val AUC {auc:.4f}")
        if auc > best_auc:
            best_auc = auc; model.save(OUT); print("   saved best model")

if EPOCHS > 0:
    train_model()
    model = SentenceTransformer(OUT, device=device)   # reload the best checkpoint
    model.max_seq_length = MAX_LEN

## 4. Results (test set)

In [ ]:
if EPOCHS > 0:
    results.append(evaluate("Fine-tuned MiniLM (this notebook)", model_scores(model, val), model_scores(model, test)))
table = pd.DataFrame(results)
print(table.to_string(index=False))
table

In [ ]:
# Average similarity of paraphrase vs non-paraphrase pairs: handy for choosing thresholds in the app
s = model_scores(model, test)
print("mean cosine, paraphrase pairs:     ", round(float(s[test.label.values == 1].mean()), 3))
print("mean cosine, non-paraphrase pairs: ", round(float(s[test.label.values == 0].mean()), 3))

# a quick sanity check on plagiarism-style examples
pairs = [("The cat sat on the mat.", "A cat was sitting on the mat."),
         ("Photosynthesis converts light energy into chemical energy.", "Plants turn sunlight into stored chemical energy."),
         ("Photosynthesis converts light energy into chemical energy.", "The stock market fell sharply on Monday.")]
for a, b in pairs:
    ea, eb = embed(model, [a, b]); print(round(float(ea @ eb), 3), "|", a, "<->", b)

## 5. Export for the app

In [ ]:
with open("/kaggle/working/metrics.json", "w") as f:
    json.dump(results, f, indent=2)
shutil.make_archive("/kaggle/working/semantic", "zip", OUT)
print("Created /kaggle/working/semantic.zip  (" + str(round(os.path.getsize('/kaggle/working/semantic.zip')/1e6)) + " MB)")
print(os.listdir(OUT))

## 6. Use it in the app
1. Notebook page -> **Output** tab -> download `semantic.zip` (and `metrics.json` for your report).
2. Unzip it **into** `plagiarism-checker/models/semantic/` so that `models/semantic/config.json` exists.
3. `pip install sentence-transformers` in the app's virtual environment.
4. Restart `python app.py`. The terminal prints `[semantic] loaded model ...` and a "Meaning" factor appears in the results.